# 02 — Parsing Portuguese Legal Documents

This notebook transforms the **page-level text extracted from PDFs** into a structured
legal corpus that can later be used to create an embedding fine-tuning dataset.

Notebook 01 answered:

> **What text can we extract from each PDF page?**

This notebook answers a different question:

> **Which parts of that text correspond to legal instruments, hierarchy levels,
> articles, paragraphs and other structural units?**

## Why parsing is necessary

An embedding model should not be trained on arbitrary PDF pages. A page is a visual
unit and may contain the end of one article and the beginning of another. Legal
retrieval is usually more meaningful when the data is organised around legal units.

The transformation is therefore approximately:

```text
Extracted pages
      ↓
Normalised lines
      ↓
Remove probable running headers/footers
      ↓
Detect legal blocks and hierarchy
      ↓
Detect articles and article boundaries
      ↓
Validate the parsed structure
      ↓
Select the intended legal corpus
      ↓
Export clean articles
```

## A central difficulty

Legal PDFs contain text that *looks structural* but is not necessarily structural.
For example, the title of a tax code may appear at the top of every page as a running
header. Treating every occurrence as the beginning of a new legal instrument would
fragment the document incorrectly.

For this reason, the parser is deliberately conservative and combines several signals:
repetition frequency, page position, explicit markers, known instrument titles,
article-number restarts and proximity to article headings.

## Learning objectives

After this notebook, you should understand:

- the difference between PDF extraction and legal parsing;
- how regular expressions can detect legal structure;
- why repeated headers and footers are problematic;
- how article-number sequences help detect document boundaries;
- why parser diagnostics are essential;
- how to preserve provenance while restructuring text;
- why suspicious records should be flagged rather than silently deleted.

> The objective is not merely to obtain *some* articles. It is to build a corpus whose
> transformation from source PDF to training data can be explained and audited.

## 1. Imports and project paths

We load the JSON documents produced by notebook 01 from `data/extracted/` and write
the structured output to `data/processed/`.

At this point the input is no longer PDF. Each document already contains a list of
pages and extracted text.

`Counter` and `defaultdict` are particularly useful here because parsing requires
frequency statistics, grouping and occurrence counting. `re` provides regular
expressions for recognising legal patterns, while `unicodedata` helps standardise
Unicode representations before matching text.

In [1]:
from pathlib import Path
from copy import deepcopy
from collections import Counter, defaultdict
import json
import re
import unicodedata

PROJECT_ROOT = Path("..")
EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(
    p for p in EXTRACTED_DIR.glob("*.json")
    if not p.name.startswith("_")
)

print(f"Documents found: {len(input_files)}")
for p in input_files:
    print(" -", p.name)

Documents found: 10
 - cimi.json
 - circ.json
 - cirs.json
 - cis.json
 - civa.json
 - ebf.json
 - lgt.json
 - oe2026.json
 - rgit.json
 - riti.json


## 2. Text normalisation

Before applying structural rules, equivalent text should have a consistent
representation.

PDF extraction may produce visually identical strings using different Unicode
representations, non-breaking spaces or irregular spacing. Such differences can make
an otherwise correct regular expression fail.

The functions below therefore create two representations:

- **normalised lines**, used while preserving readable legal text;
- **normalised search text**, used for robust case- and accent-insensitive matching.

This distinction is useful: the text stored in the corpus should remain readable,
while structural detection can use a more aggressively normalised representation.

No semantic rewriting is performed here.

In [2]:
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", text or "")


def normalize_line(line: str) -> str:
    line = normalize_unicode(line)
    line = line.replace("\u00a0", " ")
    line = re.sub(r"[ \t]+", " ", line)
    return line.strip()


def split_lines(text: str) -> list[str]:
    return [
        line
        for raw in (text or "").splitlines()
        if (line := normalize_line(raw))
    ]


def normalized_search_text(text: str) -> str:
    text = normalize_unicode(text)

    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

    return re.sub(r"\s+", " ", text).upper().strip()

## 3. Detect articles, paragraphs and letters

Portuguese legislation follows recurring textual conventions such as:

```text
Artigo 12.º
1 - ...
2 - ...
a) ...
b) ...
```

Regular expressions allow us to recognise these patterns without requiring a language
model.

The article pattern accepts common typographical variations of the ordinal symbol and
also supports suffixed articles such as `12.º-A`.

The helper functions return structured dictionaries instead of only `True`/`False`.
For example, detecting `Artigo 12.º-A` gives us both the normalised legal number and
its numeric component. The numeric component becomes useful later when analysing
article-number restarts.

A parser should tolerate harmless typographical variation while keeping the resulting
representation consistent.

In [3]:
ARTICLE_PATTERN = re.compile(
    r"^\s*Artigo\s+"
    r"(?P<number>\d+)"
    r"(?:\s*\.\s*[º°oO]|[º°])?"
    r"(?:\s*[-–—]\s*(?P<suffix>[A-Z]))?"
    r"\s*$",
    re.IGNORECASE
)

PARAGRAPH_PATTERN = re.compile(
    r"^(?P<number>\d+)\s*[-–—]\s*(?P<text>.*)$"
)

LETTER_PATTERN = re.compile(
    r"^(?P<letter>[a-z])\)\s*(?P<text>.*)$",
    re.IGNORECASE
)


def normalize_article_number(number, suffix=None):
    value = f"{int(number)}.º"

    if suffix:
        value += f"-{suffix.upper()}"

    return value


def match_article(line):
    match = ARTICLE_PATTERN.match(normalize_line(line))

    if not match:
        return None

    return {
        "number": normalize_article_number(
            match.group("number"),
            match.group("suffix")
        ),
        "numeric": int(match.group("number")),
        "suffix": (
            match.group("suffix").upper()
            if match.group("suffix")
            else None
        )
    }


def match_paragraph(line):
    match = PARAGRAPH_PATTERN.match(line)

    if not match:
        return None

    return {
        "number": match.group("number"),
        "text": normalize_line(match.group("text"))
    }


def match_letter(line):
    match = LETTER_PATTERN.match(line)

    if not match:
        return None

    return {
        "letter": match.group("letter").lower(),
        "text": normalize_line(match.group("text"))
    }

## 4. Detect the legal hierarchy

Legal instruments are often organised into levels such as:

```text
PARTE
  └── LIVRO
       └── TÍTULO
            └── CAPÍTULO
                 └── SECÇÃO
                      └── SUBSECÇÃO
```

Not every document uses every level.

`STRUCTURE_PATTERNS` recognises these headings and `HIERARCHY_ORDER` records their
relative order. When a higher-level heading changes, lower levels must eventually be
reset because they belong to the previous branch of the hierarchy.

The parser stores hierarchy information with each article. This later gives the
embedding dataset useful metadata without forcing hierarchy labels into the article
text itself.

In [4]:
ROMAN = r"[IVXLCDM]+"

STRUCTURE_PATTERNS = {
    "part": re.compile(rf"^PARTE\s+({ROMAN}|\d+.*?)\s*$", re.I),
    "book": re.compile(rf"^LIVRO\s+({ROMAN}|\d+.*?)\s*$", re.I),
    "title": re.compile(rf"^T[IÍ]TULO\s+({ROMAN}|\d+.*?)\s*$", re.I),
    "chapter": re.compile(rf"^CAP[IÍ]TULO\s+({ROMAN}|\d+.*?)\s*$", re.I),
    "section": re.compile(rf"^SEC(?:Ç|C)[AÃ]O\s+({ROMAN}|\d+.*?)\s*$", re.I),
    "subsection": re.compile(rf"^SUBSEC(?:Ç|C)[AÃ]O\s+({ROMAN}|\d+.*?)\s*$", re.I),
}

HIERARCHY_ORDER = [
    "part", "book", "title",
    "chapter", "section", "subsection"
]


def identify_structure(line):
    for kind, pattern in STRUCTURE_PATTERNS.items():
        match = pattern.match(line)

        if match:
            return {
                "type": kind,
                "value": match.group(1),
                "heading": line
            }

    return None


def empty_hierarchy():
    hierarchy = {}

    for level in HIERARCHY_ORDER:
        hierarchy[level] = None
        hierarchy[f"{level}_title"] = None

    return hierarchy

## 5. Detect legal-instrument markers

A source PDF may contain more than one legal block: the main code, annexes,
republications or additional legal instruments.

We therefore recognise candidate markers such as known Portuguese tax-law titles,
`ANEXO` and `REPUBLICAÇÃO`.

A crucial rule is:

> **Finding a legal title is evidence of a boundary, but not proof of a boundary.**

Why? Because the same title may be printed at the top of every page as a running
header.

Later sections combine marker detection with page statistics and article behaviour
before deciding whether a new block should actually be created.

In [5]:
KNOWN_INSTRUMENT_PATTERNS = [
    ("civa", r"\bCODIGO DO IMPOSTO SOBRE O VALOR ACRESCENTADO\b"),
    ("cirs", r"\bCODIGO DO IMPOSTO SOBRE O RENDIMENTO DAS PESSOAS SINGULARES\b"),
    ("circ", r"\bCODIGO DO IMPOSTO SOBRE O RENDIMENTO DAS PESSOAS COLETIVAS\b"),
    ("cimi", r"\bCODIGO DO IMPOSTO MUNICIPAL SOBRE IMOVEIS\b"),
    ("cis", r"\bCODIGO DO IMPOSTO DO SELO\b"),
    ("ebf", r"\bESTATUTO DOS BENEFICIOS FISCAIS\b"),
    ("lgt", r"\bLEI GERAL TRIBUTARIA\b"),
    ("rgit", r"\bREGIME GERAL DAS INFRACOES TRIBUTARIAS\b"),
    ("riti", r"\bREGIME DO IVA NAS TRANSACOES INTRACOMUNITARIAS(?: DE BENS)?\b"),
]

ANNEX_PATTERN = re.compile(
    r"^ANEXO(?:\s+[IVXLCDM0-9]+)?$",
    re.I
)

REPUBLICATION_PATTERN = re.compile(
    r"^(?:REPUBLICACAO|REPUBLICAÇÃO)(?:\s+DO|\s+DA|\s+DE)?(?:.*)?$",
    re.I
)


def detect_marker(line):
    searchable = normalized_search_text(line)

    for instrument, pattern in KNOWN_INSTRUMENT_PATTERNS:
        if re.search(pattern, searchable):
            return {
                "marker_type": "instrument_title",
                "instrument_candidate": instrument,
                "marker_text": line
            }

    if ANNEX_PATTERN.match(line):
        return {
            "marker_type": "annex",
            "instrument_candidate": None,
            "marker_text": line
        }

    if REPUBLICATION_PATTERN.match(line):
        return {
            "marker_type": "republication",
            "instrument_candidate": None,
            "marker_text": line
        }

    return None

## 6. Build page-line statistics

This stage measures how lines behave across the document.

For every normalised line we record:

- on how many pages it appears;
- its total number of occurrences;
- how often it appears near the top;
- how often it appears near the bottom.

This lets us distinguish content from page furniture.

For example:

```text
"Código do Imposto sobre o Valor Acrescentado"
```

appearing once before Article 1 may be a genuine document title. The same line
appearing on 80% of pages and almost always among the first five lines is much more
likely to be a running header.

This is a useful example of **document-level context**: the meaning of a line cannot
always be determined by looking at that line alone.

In [6]:
HEADER_LINE_LIMIT = 5
FOOTER_LINE_LIMIT = 5

REPEATED_PAGE_RATIO = 0.20
REPEATED_MIN_PAGES = 5


def build_line_statistics(pages):
    stats = defaultdict(lambda: {
        "pages": set(),
        "occurrences": 0,
        "top_occurrences": 0,
        "bottom_occurrences": 0
    })

    for page in pages:
        page_number = page.get("page")
        lines = split_lines(page.get("text", ""))
        n = len(lines)

        for index, line in enumerate(lines):
            item = stats[line]

            item["pages"].add(page_number)
            item["occurrences"] += 1

            if index < HEADER_LINE_LIMIT:
                item["top_occurrences"] += 1

            if n and index >= max(0, n - FOOTER_LINE_LIMIT):
                item["bottom_occurrences"] += 1

    result = {}

    for line, item in stats.items():
        result[line] = {
            "page_count": len(item["pages"]),
            "occurrences": item["occurrences"],
            "top_occurrences": item["top_occurrences"],
            "bottom_occurrences": item["bottom_occurrences"]
        }

    return result


def is_probable_running_header(
    line,
    stats,
    total_pages
):
    item = stats.get(line)

    if not item:
        return False

    threshold = max(
        REPEATED_MIN_PAGES,
        int(total_pages * REPEATED_PAGE_RATIO)
    )

    repeated = item["page_count"] >= threshold

    positional = (
        item["top_occurrences"] >= max(3, item["page_count"] * 0.50)
        or
        item["bottom_occurrences"] >= max(3, item["page_count"] * 0.50)
    )

    return repeated and positional

## 7. Inspect repeated legal-title markers

Before automatically removing anything, we inspect how detected legal titles behave
in each source document.

This diagnostic displays the candidate instrument, occurrence statistics and whether
the line is classified as a probable running header.

During a training session, this is a useful point to emphasise that parser development
is iterative:

```text
write rule → inspect diagnostics → identify failure → refine rule
```

Legal-document parsing should not be treated as a single opaque transformation.

In [7]:
for input_path in input_files:
    with input_path.open("r", encoding="utf-8") as f:
        document = json.load(f)

    pages = document.get("pages", [])
    stats = build_line_statistics(pages)

    print(f"\n{'=' * 80}")
    print(document.get("document_id", input_path.stem).upper())

    shown = set()

    for page in pages:
        for line in split_lines(page.get("text", "")):
            marker = detect_marker(line)

            if (
                marker
                and marker["marker_type"] == "instrument_title"
                and line not in shown
            ):
                shown.add(line)

                print({
                    "text": line,
                    "candidate": marker["instrument_candidate"],
                    "stats": stats.get(line),
                    "running_header": is_probable_running_header(
                        line,
                        stats,
                        len(pages)
                    )
                })


CIMI
{'text': 'Código do Imposto Municipal sobre Imóveis (CIMI)', 'candidate': 'cimi', 'stats': {'page_count': 89, 'occurrences': 89, 'top_occurrences': 89, 'bottom_occurrences': 2}, 'running_header': True}
{'text': 'artigo 26.o do Código do Imposto do Selo, consoante o caso. (Redação do DL 211/2005, de 7 de', 'candidate': 'cis', 'stats': {'page_count': 1, 'occurrences': 1, 'top_occurrences': 0, 'bottom_occurrences': 0}, 'running_header': False}
{'text': 'autárquica consideram-se referidos ao Código do Imposto Municipal sobre Imóveis (CIMI) ou ao', 'candidate': 'cimi', 'stats': {'page_count': 1, 'occurrences': 1, 'top_occurrences': 0, 'bottom_occurrences': 0}, 'running_header': False}
{'text': 'Onerosas de Imóveis (CIMT), ao Código do Imposto do Selo, ao imposto municipal sobre as', 'candidate': 'cis', 'stats': {'page_count': 1, 'occurrences': 1, 'top_occurrences': 0, 'bottom_occurrences': 0}, 'running_header': False}
{'text': 'CIMT e pelo Código do Imposto do Selo são aprovados por p

## 8. Remove probable running headers and footers

Repeated page furniture can contaminate article text and, more importantly, can be
mistaken for legal structure.

The code first identifies repeated lines and then removes them when individual pages
are cleaned.

Two safeguards are important:

1. lines beginning with `Artigo` are preserved;
2. explicit annex/republication markers are preserved.

The goal is not to remove every repeated line. It is to remove lines for which the
document-level evidence strongly suggests that they are page headers or footers.

In [8]:
ARTICLE_WORD_PATTERN = re.compile(r"^\s*artigo\b", re.I)


def detect_repeated_page_lines(pages):
    stats = build_line_statistics(pages)
    total_pages = len(pages)

    repeated = set()

    for line in stats:
        if ARTICLE_WORD_PATTERN.match(line):
            continue

        marker = detect_marker(line)

        # Structural annex/republication markers are preserved.
        if marker and marker["marker_type"] in {
            "annex", "republication"
        }:
            continue

        if is_probable_running_header(
            line,
            stats,
            total_pages
        ):
            repeated.add(line)

    return repeated


def clean_page_lines(text, repeated_lines):
    return [
        line
        for line in split_lines(text)
        if line not in repeated_lines
    ]

## 9. Detect article-number restarts

Article numbering provides another structural signal.

Consider:

```text
... Artigo 80.º
... Artigo 81.º
ANEXO
Artigo 1.º
```

A jump from a high article number back to Article 1 may indicate that a new legal
block has started.

The configured thresholds deliberately avoid treating every small backward movement
as a boundary. Number restarts are therefore one signal among several rather than an
absolute rule.

This illustrates a recurring parser-design principle: combine weak signals instead of
relying on one brittle heuristic.

In [9]:
BLOCK_CONFIG = {
    "restart_article_max": 3,
    "previous_article_min": 5,
    "minimum_backward_jump": 10
}


def is_number_restart(previous, current):
    if previous is None:
        return False

    previous_number = previous["numeric"]
    current_number = current["numeric"]

    if (
        current_number == 1
        and previous_number >= BLOCK_CONFIG["previous_article_min"]
    ):
        return True

    return (
        current_number <= BLOCK_CONFIG["restart_article_max"]
        and previous_number >= BLOCK_CONFIG["previous_article_min"]
        and
        (previous_number - current_number)
        >= BLOCK_CONFIG["minimum_backward_jump"]
    )

## 10. Pending structural context

Markers expire after a configurable number of meaningful lines. This prevents a
title seen far away from an article from incorrectly affecting a later block.

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [10]:
MARKER_MAX_DISTANCE = 12


def new_pending_marker(marker, page, line_index):
    return {
        **marker,
        "marker_page": page,
        "marker_line_index": line_index,
        "distance": 0
    }


def advance_marker(marker):
    if marker is None:
        return None

    marker = dict(marker)
    marker["distance"] += 1

    if marker["distance"] > MARKER_MAX_DISTANCE:
        return None

    return marker


def can_be_heading_title(line):
    return (
        bool(line)
        and not identify_structure(line)
        and not match_article(line)
        and not match_paragraph(line)
        and not match_letter(line)
        and not detect_marker(line)
        and len(line) <= 180
    )

## 11. Block and article constructors

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [11]:
def create_block(
    document_id,
    index,
    page,
    reason,
    marker=None
):
    return {
        "block_index": index,
        "block_id": f"{document_id}_block_{index:02d}",
        "page_start": page,
        "page_end": page,
        "start_reason": reason,
        "marker_type": marker.get("marker_type") if marker else None,
        "marker_text": marker.get("marker_text") if marker else None,
        "marker_page": marker.get("marker_page") if marker else None,
        "instrument_candidate": (
            marker.get("instrument_candidate")
            if marker else None
        )
    }


def create_article(
    document_id,
    block,
    article_match,
    page,
    hierarchy
):
    return {
        "document_id": document_id,
        "block_index": block["block_index"],
        "block_id": block["block_id"],
        "instrument_candidate": block.get("instrument_candidate"),
        "article": article_match["number"],
        "article_numeric": article_match["numeric"],
        "article_suffix": article_match["suffix"],
        "article_title": None,
        **deepcopy(hierarchy),
        "page_start": page,
        "page_end": page,
        "lines": [],
        "paragraphs": []
    }

## 12. v5 block-decision logic

A new block starts under one of these conditions:

### A. First detected article
Always starts block 1.

### B. Strong numeric restart
For example `Article 16.º → Article 1.º`.

### C. Explicit annex/republication context
A nearby `ANEXO` or `REPUBLICAÇÃO`, combined with an article number near the
start of a legal instrument (`1.º`, `2.º`, or `3.º`).

### D. Specific legal title plus corroborating evidence
A non-header legal title can start a new block only if:

- the next article is near the beginning (`1.º`–`3.º`); and
- there is either a backwards numbering movement or an annex/republication
  context.

A repeated page header can therefore no longer create a new block.

In [12]:
def decide_block_start(
    current_block,
    previous_article,
    current_article,
    pending_marker,
    pending_structural_marker
):
    if current_block is None:
        return True, "first_article"

    number_restart = is_number_restart(
        previous_article,
        current_article
    )

    if number_restart:
        return True, "number_restart"

    near_start = current_article["numeric"] <= 3

    if (
        pending_structural_marker is not None
        and near_start
    ):
        return True, (
            f"{pending_structural_marker['marker_type']}"
            "_and_low_article"
        )

    if (
        pending_marker is not None
        and pending_marker["marker_type"] == "instrument_title"
        and near_start
        and previous_article is not None
        and current_article["numeric"] < previous_article["numeric"]
    ):
        return True, "instrument_title_and_backward_numbering"

    return False, None

## 13. v5 parser

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [13]:
def parse_legal_document(document):
    document_id = (
        document.get("document_id")
        or Path(document.get("source_file", "document")).stem.lower()
    )

    pages = document.get("pages", [])
    total_pages = len(pages)

    line_stats = build_line_statistics(pages)
    repeated_lines = detect_repeated_page_lines(pages)

    hierarchy = empty_hierarchy()

    blocks = []
    articles = []

    current_block = None
    current_article = None
    current_paragraph = None
    current_letter = None

    previous_article_match = None

    pending_instrument_marker = None
    pending_structural_marker = None

    pending_structure = None
    waiting_for_article_title = False

    def close_article():
        nonlocal current_article

        if current_article is None:
            return

        current_article["text"] = "\n".join(
            current_article.pop("lines")
        ).strip()

        articles.append(current_article)
        current_article = None

    def start_block(page, reason, marker=None):
        nonlocal current_block, hierarchy

        if current_block is not None:
            current_block["page_end"] = page

        current_block = create_block(
            document_id=document_id,
            index=len(blocks) + 1,
            page=page,
            reason=reason,
            marker=marker
        )

        blocks.append(current_block)
        hierarchy = empty_hierarchy()

    for page in pages:
        page_number = page.get("page")
        raw_lines = split_lines(page.get("text", ""))

        # Keep original indices for positional diagnostics.
        filtered = [
            (idx, line)
            for idx, line in enumerate(raw_lines)
            if line not in repeated_lines
        ]

        i = 0

        while i < len(filtered):
            original_index, line = filtered[i]

            marker = detect_marker(line)

            if marker:
                running_header = is_probable_running_header(
                    line,
                    line_stats,
                    total_pages
                )

                if not running_header:
                    pending = new_pending_marker(
                        marker,
                        page_number,
                        original_index
                    )

                    if marker["marker_type"] == "instrument_title":
                        pending_instrument_marker = pending

                    elif marker["marker_type"] in {
                        "annex",
                        "republication"
                    }:
                        pending_structural_marker = pending

                i += 1
                continue

            pending_instrument_marker = advance_marker(
                pending_instrument_marker
            )

            pending_structural_marker = advance_marker(
                pending_structural_marker
            )

            if pending_structure is not None:
                if can_be_heading_title(line):
                    hierarchy[
                        f"{pending_structure}_title"
                    ] = line

                    pending_structure = None
                    i += 1
                    continue

                pending_structure = None

            if (
                waiting_for_article_title
                and current_article is not None
            ):
                if can_be_heading_title(line):
                    current_article["article_title"] = line
                    waiting_for_article_title = False
                    i += 1
                    continue

                waiting_for_article_title = False

            structure = identify_structure(line)

            if structure:
                level = structure["type"]

                hierarchy[level] = structure["value"]
                hierarchy[f"{level}_title"] = None

                level_index = HIERARCHY_ORDER.index(level)

                for lower in HIERARCHY_ORDER[level_index + 1:]:
                    hierarchy[lower] = None
                    hierarchy[f"{lower}_title"] = None

                pending_structure = level

                i += 1
                continue

            article_match = match_article(line)

            if article_match:
                should_start, reason = decide_block_start(
                    current_block,
                    previous_article_match,
                    article_match,
                    pending_instrument_marker,
                    pending_structural_marker
                )

                close_article()

                if should_start:
                    marker_for_block = (
                        pending_instrument_marker
                        or pending_structural_marker
                    )

                    start_block(
                        page=page_number,
                        reason=reason,
                        marker=marker_for_block
                    )

                current_article = create_article(
                    document_id,
                    current_block,
                    article_match,
                    page_number,
                    hierarchy
                )

                current_paragraph = None
                current_letter = None
                waiting_for_article_title = True

                previous_article_match = article_match

                pending_instrument_marker = None
                pending_structural_marker = None

                i += 1
                continue

            if current_article is None:
                i += 1
                continue

            current_article["page_end"] = page_number

            if current_block is not None:
                current_block["page_end"] = page_number

            paragraph = match_paragraph(line)

            if paragraph:
                current_paragraph = {
                    "number": paragraph["number"],
                    "page": page_number,
                    "text": paragraph["text"],
                    "letters": []
                }

                current_article["paragraphs"].append(
                    current_paragraph
                )

                current_letter = None
                current_article["lines"].append(line)

                i += 1
                continue

            letter = match_letter(line)

            if letter:
                current_letter = {
                    "letter": letter["letter"],
                    "page": page_number,
                    "text": letter["text"]
                }

                if current_paragraph is not None:
                    current_paragraph["letters"].append(
                        current_letter
                    )

                current_article["lines"].append(line)

                i += 1
                continue

            current_article["lines"].append(line)

            if current_letter is not None:
                current_letter["text"] = (
                    current_letter["text"]
                    + " "
                    + line
                ).strip()

            elif current_paragraph is not None:
                current_paragraph["text"] = (
                    current_paragraph["text"]
                    + " "
                    + line
                ).strip()

            i += 1

    close_article()

    if current_block is not None and pages:
        current_block["page_end"] = pages[-1].get("page")

    return {
        "document_id": document_id,
        "document_name": document.get("document_name"),
        "source_file": document.get("source_file"),
        "total_pages": document.get(
            "total_pages",
            len(pages)
        ),
        "parser_version": "5.0",
        "block_count": len(blocks),
        "article_count": len(articles),
        "repeated_lines_removed": sorted(repeated_lines),
        "blocks": blocks,
        "articles": articles
    }

## 14. Parse all documents

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [14]:
parsed_documents = []

for input_path in input_files:
    with input_path.open("r", encoding="utf-8") as f:
        document = json.load(f)

    parsed = parse_legal_document(document)
    parsed_documents.append(parsed)

    print(
        f"{parsed['document_id'].upper():10} | "
        f"{parsed['block_count']:>3} blocks | "
        f"{parsed['article_count']:>4} articles"
    )

CIMI       |   2 blocks |  189 articles
CIRC       |   1 blocks |  171 articles
CIRS       |   2 blocks |  216 articles
CIS        |   2 blocks |   93 articles
CIVA       |   3 blocks |  218 articles
EBF        |   5 blocks |  183 articles
LGT        |   2 blocks |  124 articles
OE2026     |   1 blocks |  277 articles
RGIT       |   2 blocks |  147 articles
RITI       |   4 blocks |  150 articles


## 15. Parse documents into legal blocks and articles

This is the central transformation of the notebook.

The parser walks through each cleaned page line by line while maintaining **state**.
At any moment it needs to remember information such as:

```text
current legal block
current hierarchy
current article
previous article number
pending structural marker
```

This is sometimes called a **stateful parser**.

When an article heading is detected, the previous article can be closed and a new
article opened. When hierarchy headings are detected, the hierarchy state is updated.
When sufficiently strong boundary evidence is found, a new legal block is created.

The important point is that an article is reconstructed from information that may span
multiple lines and pages. Parsing therefore cannot be reduced to searching each page
independently.

In [15]:
block_summary = []

for document in parsed_documents:
    grouped = defaultdict(list)

    for article in document["articles"]:
        grouped[article["block_id"]].append(article)

    print(f"\n{'=' * 90}")
    print(document["document_id"].upper())

    for block in document["blocks"]:
        articles = grouped[block["block_id"]]
        numbers = [a["article"] for a in articles]

        row = {
            "document_id": document["document_id"],
            "block_id": block["block_id"],
            "pages": (
                f"{block['page_start']}-"
                f"{block['page_end']}"
            ),
            "reason": block["start_reason"],
            "instrument_candidate": (
                block["instrument_candidate"]
            ),
            "marker_text": block["marker_text"],
            "articles": len(articles),
            "first_article": (
                numbers[0] if numbers else None
            ),
            "last_article": (
                numbers[-1] if numbers else None
            )
        }

        block_summary.append(row)
        print(row)


CIMI
{'document_id': 'cimi', 'block_id': 'cimi_block_01', 'pages': '8-25', 'reason': 'first_article', 'instrument_candidate': None, 'marker_text': None, 'articles': 34, 'first_article': '13.º', 'last_article': '32.º'}
{'document_id': 'cimi', 'block_id': 'cimi_block_02', 'pages': '25-91', 'reason': 'number_restart', 'instrument_candidate': None, 'marker_text': None, 'articles': 155, 'first_article': '1.º', 'last_article': '140.º'}

CIRC
{'document_id': 'circ', 'block_id': 'circ_block_01', 'pages': '17-164', 'reason': 'first_article', 'instrument_candidate': None, 'marker_text': None, 'articles': 171, 'first_article': '1.º', 'last_article': '143.º'}

CIRS
{'document_id': 'cirs', 'block_id': 'cirs_block_01', 'pages': '7-23', 'reason': 'first_article', 'instrument_candidate': 'cirs', 'marker_text': 'Aprova o Código do Imposto sobre o Rendimento das Pessoas Singulares, anexo ao presente', 'articles': 17, 'first_article': '1.º', 'last_article': '16.º'}
{'document_id': 'cirs', 'block_id': 'c

## 16. Parse the complete extracted corpus

We now apply the parser to every JSON document produced by notebook 01.

Each parsed document contains the reconstructed articles together with metadata such
as source document, legal block, hierarchy and page range.

The resulting files are intermediate parser outputs. They should still be validated
before becoming training data.

A successful execution means the algorithm completed; it does **not** yet mean that
all detected article boundaries are legally correct.

In [16]:
def slugify_identifier(value):
    value = normalize_unicode(value).lower()
    value = value.replace(".º", "")
    value = value.replace("º", "")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def build_article_id(article):
    return (
        f"{article['block_id']}_"
        f"art_{slugify_identifier(article['article'])}"
    )


for document in parsed_documents:
    for article in document["articles"]:
        article["article_id"] = build_article_id(article)

## 17. Inspect block-level diagnostics

Legal-block diagnostics help answer questions such as:

- How many blocks were detected?
- Which marker triggered each block?
- Where did article numbering restart?
- Did a running header accidentally create many tiny blocks?

A plausible parser should produce a small, explainable number of blocks. Unexpected
fragmentation is usually a sign that one of the structural heuristics is too
aggressive.

In [17]:
for document in parsed_documents:
    print(f"\n{'=' * 80}")
    print(document["document_id"].upper())

    grouped = defaultdict(list)

    for article in document["articles"]:
        grouped[article["block_id"]].append(article)

    for block_id, articles in grouped.items():
        numbers = [a["article"] for a in articles]

        print(
            f"\n{block_id} | "
            f"{len(numbers)} article(s)"
        )
        print("First:", numbers[:20])
        print("Last :", numbers[-20:])


CIMI

cimi_block_01 | 34 article(s)
First: ['13.º', '14.º', '15.º', '15.º-A', '15.º-B', '15.º-C', '15.º-D', '15.º-E', '15.º-F', '15.º-G', '15.º-H', '15.º-I', '15.º-J', '15.º-L', '15.º-M', '15.º-N', '15.º-O', '15.º-P', '16.º', '18.º']
Last : ['15.º-M', '15.º-N', '15.º-O', '15.º-P', '16.º', '18.º', '19.º', '20.º', '21.º', '22.º', '23.º', '24.º', '25.º', '26.º', '27.º', '28.º', '29.º', '30.º', '31.º', '32.º']

cimi_block_02 | 155 article(s)
First: ['1.º', '2.º', '3.º', '4.º', '5.º', '6.º', '7.º', '8.º', '9.º', '10.º', '11.º', '11.º-A', '11.º-B', '12.º', '13.º', '13.º-A', '14.º', '15.º', '16.º', '17.º']
Last : ['133.º', '134.º', '135.º', '135.º-A', '135.º-B', '135.º-C', '135.º-D', '135.º-E', '135.º-F', '135.º-G', '135.º-H', '135.º-I', '135.º-J', '135.º-K', '135.º-L', '135.º-M', '136.º', '137.º', '138.º', '140.º']

CIRC

circ_block_01 | 171 article(s)
First: ['1.º', '2.º', '3.º', '4.º', '5.º', '6.º', '7.º', '8.º', '9.º', '10.º', '11.º', '12.º', '13.º', '14.º', '15.º', '16.º', '17.º', '18.º

## 18. Inspect article sequences

Article numbering is one of the strongest sanity checks available in legislation.

Within the intended legal instrument, we normally expect a broadly coherent sequence.
Unexpected duplicates, large backward jumps or unusual gaps may indicate:

- a missed block boundary;
- an annex absorbed into the main code;
- an article heading that was not detected;
- a false article heading;
- legitimate amendments or special numbering that require manual interpretation.

The purpose of this diagnostic is therefore not to demand perfect numerical
continuity, but to expose suspicious transitions for review.

In [18]:
duplicate_report = []

for document in parsed_documents:
    grouped = defaultdict(list)

    for article in document["articles"]:
        grouped[article["block_id"]].append(article)

    for block_id, articles in grouped.items():
        counts = Counter(
            a["article"]
            for a in articles
        )

        duplicates = {
            number: count
            for number, count in counts.items()
            if count > 1
        }

        if duplicates:
            duplicate_report.append({
                "document_id": document["document_id"],
                "block_id": block_id,
                "duplicates": duplicates
            })

            print(block_id, duplicates)

print(
    f"\nBlocks with duplicates: "
    f"{len(duplicate_report)}"
)

cimi_block_02 {'25.º': 2}
circ_block_01 {'73.º': 2, '91.º': 2}
cirs_block_02 {'11.º': 2, '103.º': 2, '128.º': 2}
civa_block_01 {'2.º': 2}
civa_block_02 {'5.º': 2, '6.º': 2}
ebf_block_01 {'2.º': 2}
ebf_block_03 {'1.º': 2, '4.º': 2}
oe2026_block_01 {'27.º': 2, '52.º': 2, '61.º': 2, '68.º': 2, '70.º': 2, '88.º': 2, '76.º': 2, '94.º': 2, '101.º': 2, '106.º': 2, '108.º': 2, '109.º': 2, '115.º': 2}
rgit_block_02 {'30.º': 2, '31.º': 2, '32.º': 2, '40.º': 2, '70.º': 2, '75.º': 2, '79.º': 2, '80.º': 2, '83.º': 2, '84.º': 2, '104.º': 2}
riti_block_02 {'2.º': 2}
riti_block_03 {'5.º': 2, '6.º': 2}

Blocks with duplicates: 11


## 19. Inspect hierarchy assignments

A correctly detected article is more useful when its legal context is also known.

For selected articles we inspect the hierarchy captured when the article was opened,
for example its title, chapter or section.

This metadata can later support filtering, analysis and richer retrieval interfaces.
It also provides another way of detecting parser errors: an implausible hierarchy
transition may reveal that a heading was missed or misclassified.

In [19]:
LARGE_ARTICLE_THRESHOLD = 15000

large_article_report = []

for document in parsed_documents:
    for article in document["articles"]:
        size = len(article.get("text", ""))

        if size > LARGE_ARTICLE_THRESHOLD:
            large_article_report.append({
                "document_id": document["document_id"],
                "block_id": article["block_id"],
                "article": article["article"],
                "article_id": article["article_id"],
                "characters": size,
                "page_start": article["page_start"],
                "page_end": article["page_end"]
            })

print(
    f"Suspiciously large articles: "
    f"{len(large_article_report)}"
)

for item in large_article_report:
    print(item)

Suspiciously large articles: 7
{'document_id': 'cirs', 'block_id': 'cirs_block_01', 'article': '16.º', 'article_id': 'cirs_block_01_art_16_o', 'characters': 44348, 'page_start': 11, 'page_end': 22}
{'document_id': 'cirs', 'block_id': 'cirs_block_02', 'article': '10.º', 'article_id': 'cirs_block_02_art_10_o', 'characters': 23012, 'page_start': 34, 'page_end': 40}
{'document_id': 'cirs', 'block_id': 'cirs_block_02', 'article': '72.º', 'article_id': 'cirs_block_02_art_72_o', 'characters': 19174, 'page_start': 86, 'page_end': 91}
{'document_id': 'civa', 'block_id': 'civa_block_03', 'article': '6.º', 'article_id': 'civa_block_03_art_6_o', 'characters': 22955, 'page_start': 81, 'page_end': 87}
{'document_id': 'civa', 'block_id': 'civa_block_03', 'article': '102.º', 'article_id': 'civa_block_03_art_102_o', 'characters': 41512, 'page_start': 170, 'page_end': 188}
{'document_id': 'oe2026', 'block_id': 'oe2026_block_01', 'article': '8.º', 'article_id': 'oe2026_block_01_art_8_o', 'characters': 18

## 20. Inspect article text and page provenance

Structural statistics should always be complemented with direct inspection of the
actual reconstructed text.

For sample articles, verify:

- article number and title;
- beginning and end of the provision;
- paragraph and letter continuity;
- page range;
- absence of obvious running headers;
- absence of text belonging to the following article.

The page range is particularly important because it keeps the structured article
traceable to the original PDF.

In [20]:
article_ids = [
    article["article_id"]
    for document in parsed_documents
    for article in document["articles"]
]

duplicate_ids = [
    article_id
    for article_id, count in Counter(article_ids).items()
    if count > 1
]

print(f"Total IDs     : {len(article_ids)}")
print(f"Unique IDs    : {len(set(article_ids))}")
print(f"Duplicate IDs : {len(duplicate_ids)}")

if duplicate_ids:
    print(duplicate_ids[:100])

Total IDs     : 1768
Unique IDs    : 1729
Duplicate IDs : 39
['cimi_block_02_art_25_o', 'circ_block_01_art_73_o', 'circ_block_01_art_91_o', 'cirs_block_02_art_11_o', 'cirs_block_02_art_103_o', 'cirs_block_02_art_128_o', 'civa_block_01_art_2_o', 'civa_block_02_art_5_o', 'civa_block_02_art_6_o', 'ebf_block_01_art_2_o', 'ebf_block_03_art_1_o', 'ebf_block_03_art_4_o', 'oe2026_block_01_art_27_o', 'oe2026_block_01_art_52_o', 'oe2026_block_01_art_61_o', 'oe2026_block_01_art_68_o', 'oe2026_block_01_art_70_o', 'oe2026_block_01_art_88_o', 'oe2026_block_01_art_76_o', 'oe2026_block_01_art_94_o', 'oe2026_block_01_art_101_o', 'oe2026_block_01_art_106_o', 'oe2026_block_01_art_108_o', 'oe2026_block_01_art_109_o', 'oe2026_block_01_art_115_o', 'rgit_block_02_art_30_o', 'rgit_block_02_art_31_o', 'rgit_block_02_art_32_o', 'rgit_block_02_art_40_o', 'rgit_block_02_art_70_o', 'rgit_block_02_art_75_o', 'rgit_block_02_art_79_o', 'rgit_block_02_art_80_o', 'rgit_block_02_art_83_o', 'rgit_block_02_art_84_o', 'rgi

## 21. Why corpus selection is a separate stage

Parsing discovers structure. Corpus selection decides **which discovered structure is
appropriate for the machine-learning experiment**.

These are different responsibilities.

A PDF can legitimately contain several blocks, but the fine-tuning experiment may
intend to use only the principal consolidated code. Hard-coding that decision deep
inside the parser would make the parser less reusable and the experiment harder to
audit.

We therefore first parse broadly and then explicitly select the target blocks.

In [21]:
main_code_candidates = []

for document in parsed_documents:
    candidates = [
        block["block_id"]
        for block in document["blocks"]
        if (
            block.get("instrument_candidate")
            == document["document_id"]
        )
    ]

    main_code_candidates.append({
        "document_id": document["document_id"],
        "candidate_blocks": candidates
    })

main_code_candidates

[{'document_id': 'cimi', 'candidate_blocks': []},
 {'document_id': 'circ', 'candidate_blocks': []},
 {'document_id': 'cirs',
  'candidate_blocks': ['cirs_block_01', 'cirs_block_02']},
 {'document_id': 'cis', 'candidate_blocks': []},
 {'document_id': 'civa', 'candidate_blocks': []},
 {'document_id': 'ebf', 'candidate_blocks': ['ebf_block_03', 'ebf_block_05']},
 {'document_id': 'lgt', 'candidate_blocks': []},
 {'document_id': 'oe2026', 'candidate_blocks': []},
 {'document_id': 'rgit', 'candidate_blocks': []},
 {'document_id': 'riti',
  'candidate_blocks': ['riti_block_01', 'riti_block_04']}]

## 22. Define the corpus configuration

The corpus configuration records which legal block from each source document is
considered the main training corpus.

This is an explicit experimental decision and should be reviewed whenever the source
PDFs change.

Keeping this mapping in one visible configuration object is preferable to scattering
document-specific exceptions throughout the parsing code.

In [22]:
def inspect_block(
    document,
    block_id,
    max_articles=6,
    max_chars=1000
):
    block = next(
        b for b in document["blocks"]
        if b["block_id"] == block_id
    )

    articles = [
        a for a in document["articles"]
        if a["block_id"] == block_id
    ]

    print(
        json.dumps(
            block,
            ensure_ascii=False,
            indent=2
        )
    )

    for article in articles[:max_articles]:
        print("\n" + "-" * 70)
        print(
            article["article_id"],
            "|",
            article.get("article_title")
        )
        print(
            article.get("text", "")[:max_chars]
        )


# Example:
# inspect_block(parsed_documents[0], "cimi_block_01")

## 23. Validate configured blocks

Before selecting articles, we verify that every configured document and block actually
exists in the parser output.

Failing early here is preferable to silently producing an incomplete corpus.

This is a general reproducibility principle: assumptions that can be checked
programmatically should be checked before expensive downstream processing begins.

In [23]:
quality_report = []

for document in parsed_documents:
    duplicate_blocks = [
        item
        for item in duplicate_report
        if item["document_id"] == document["document_id"]
    ]

    large_articles = [
        item
        for item in large_article_report
        if item["document_id"] == document["document_id"]
    ]

    candidate_blocks = [
        block["block_id"]
        for block in document["blocks"]
        if (
            block.get("instrument_candidate")
            == document["document_id"]
        )
    ]

    quality_report.append({
        "document_id": document["document_id"],
        "pages": document["total_pages"],
        "blocks": document["block_count"],
        "articles": document["article_count"],
        "candidate_main_blocks": candidate_blocks,
        "blocks_with_duplicates": len(duplicate_blocks),
        "large_articles": len(large_articles)
    })

quality_report

[{'document_id': 'cimi',
  'pages': 91,
  'blocks': 2,
  'articles': 189,
  'candidate_main_blocks': [],
  'blocks_with_duplicates': 1,
  'large_articles': 0},
 {'document_id': 'circ',
  'pages': 164,
  'blocks': 1,
  'articles': 171,
  'candidate_main_blocks': [],
  'blocks_with_duplicates': 1,
  'large_articles': 0},
 {'document_id': 'cirs',
  'pages': 139,
  'blocks': 2,
  'articles': 216,
  'candidate_main_blocks': ['cirs_block_01', 'cirs_block_02'],
  'blocks_with_duplicates': 1,
  'large_articles': 3},
 {'document_id': 'cis',
  'pages': 64,
  'blocks': 2,
  'articles': 93,
  'candidate_main_blocks': [],
  'blocks_with_duplicates': 0,
  'large_articles': 0},
 {'document_id': 'civa',
  'pages': 188,
  'blocks': 3,
  'articles': 218,
  'candidate_main_blocks': [],
  'blocks_with_duplicates': 2,
  'large_articles': 2},
 {'document_id': 'ebf',
  'pages': 107,
  'blocks': 5,
  'articles': 183,
  'candidate_main_blocks': ['ebf_block_03', 'ebf_block_05'],
  'blocks_with_duplicates': 2,
 

## 24. Select articles from the intended legal blocks

Only articles belonging to the explicitly configured main blocks are selected.

At this stage, selection does not yet mean that every article is safe for training.
The selected records still pass through duplicate and size diagnostics below.

The distinction is:

```text
parsed article
    ↓
belongs to intended block?
    ↓ yes
selected article
    ↓
passes validation checks?
    ↓ yes
ready article
```

In [24]:
for document, input_path in zip(
    parsed_documents,
    input_files
):
    output_path = (
        PROCESSED_DIR / input_path.name
    )

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            document,
            f,
            ensure_ascii=False,
            indent=2
        )


articles_path = (
    PROCESSED_DIR / "articles.jsonl"
)

with articles_path.open(
    "w",
    encoding="utf-8"
) as f:
    for document in parsed_documents:
        for article in document["articles"]:
            f.write(
                json.dumps(
                    article,
                    ensure_ascii=False
                )
                + "\n"
            )


reports = {
    "_block_summary.json": block_summary,
    "_duplicate_report.json": duplicate_report,
    "_large_article_report.json": large_article_report,
    "_main_code_candidates.json": main_code_candidates,
    "_quality_report.json": quality_report
}

for filename, data in reports.items():
    path = PROCESSED_DIR / filename

    with path.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )


print(f"Corpus: {articles_path.resolve()}")

for filename in reports:
    print(
        f"Report: "
        f"{(PROCESSED_DIR / filename).resolve()}"
    )

Corpus: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\articles.jsonl
Report: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_block_summary.json
Report: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_duplicate_report.json
Report: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_large_article_report.json
Report: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_main_code_candidates.json
Report: C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_quality_report.json


## 25. Validation criteria before dataset generation

The parser is ready for the next stage only when the outputs are structurally
plausible.

Review the following:

1. block counts should be small and interpretable;
2. repeated code titles should no longer create one block per page;
3. approval legislation and the actual code should be separated where possible;
4. article numbering inside each block should be coherent;
5. duplicate article IDs should be zero;
6. duplicate article numbers inside candidate main-code blocks should be zero;
7. unusually large articles should be manually checked;
8. candidate main-code blocks should correspond to the intended legal corpus.

If these checks are satisfactory, `03_dataset.ipynb` can select the validated
blocks and construct training, validation and evaluation examples.

# 26. Controlled corpus construction (v6)

Version 6 keeps the v5 parsing logic but changes the downstream strategy.

Instead of trying to infer the legally relevant block with increasingly complex
heuristics, the corpus is selected explicitly. This is appropriate for a
fine-tuning project because the training corpus should be controlled and
auditable.

The configuration below should be treated as a **reviewable corpus manifest**.
The proposed block choices are based on the v5 diagnostics and can be changed
without changing the parser itself.

In [25]:
CORPUS_CONFIG = {
    "cimi":   {"main_block": "cimi_block_02"},
    "circ":   {"main_block": "circ_block_01"},
    "cirs":   {"main_block": "cirs_block_02"},
    "cis":    {"main_block": "cis_block_02"},
    "civa":   {"main_block": "civa_block_03"},
    "ebf":    {"main_block": "ebf_block_05"},
    "lgt":    {"main_block": "lgt_block_02"},
    "rgit":   {"main_block": "rgit_block_02"},
    "riti":   {"main_block": "riti_block_04"},
    "oe2026": {"main_block": "oe2026_block_01"},
}

print("Configured corpus blocks:")
for document_id, config in CORPUS_CONFIG.items():
    print(f" - {document_id:8} -> {config['main_block']}")

Configured corpus blocks:
 - cimi     -> cimi_block_02
 - circ     -> circ_block_01
 - cirs     -> cirs_block_02
 - cis      -> cis_block_02
 - civa     -> civa_block_03
 - ebf      -> ebf_block_05
 - lgt      -> lgt_block_02
 - rgit     -> rgit_block_02
 - riti     -> riti_block_04
 - oe2026   -> oe2026_block_01


## 27. Validate the corpus manifest

This check verifies that every configured document and block actually exists in
the current v5 parsing output. Missing blocks are reported explicitly.

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [26]:
documents_by_id = {
    document["document_id"]: document
    for document in parsed_documents
}

manifest_errors = []

for document_id, config in CORPUS_CONFIG.items():
    document = documents_by_id.get(document_id)

    if document is None:
        manifest_errors.append(
            f"Document not found: {document_id}"
        )
        continue

    available_blocks = {
        block["block_id"]
        for block in document["blocks"]
    }

    if config["main_block"] not in available_blocks:
        manifest_errors.append(
            f"{document_id}: block not found -> "
            f"{config['main_block']}"
        )

if manifest_errors:
    print("Manifest validation FAILED:")
    for error in manifest_errors:
        print(" -", error)
else:
    print("Manifest validation passed.")

Manifest validation passed.


## 28. Select only validated legal blocks

All other embedded legislation remains available in the full processed JSON,
but it is excluded from the fine-tuning corpus.

This stage is kept separate so that its assumptions and outputs can be inspected before they influence downstream dataset construction.

In [27]:
selected_articles = []

for document_id, config in CORPUS_CONFIG.items():
    document = documents_by_id.get(document_id)

    if document is None:
        continue

    target_block = config["main_block"]

    block_articles = [
        article
        for article in document["articles"]
        if article["block_id"] == target_block
    ]

    selected_articles.extend(block_articles)

    print(
        f"{document_id.upper():8} | "
        f"{target_block:20} | "
        f"{len(block_articles):4} articles"
    )

print(f"\nSelected articles: {len(selected_articles)}")

CIMI     | cimi_block_02        |  155 articles
CIRC     | circ_block_01        |  171 articles
CIRS     | cirs_block_02        |  199 articles
CIS      | cis_block_02         |   73 articles
CIVA     | civa_block_03        |  102 articles
EBF      | ebf_block_05         |  122 articles
LGT      | lgt_block_02         |  118 articles
RGIT     | rgit_block_02        |  145 articles
RITI     | riti_block_04        |   33 articles
OE2026   | oe2026_block_01      |  277 articles

Selected articles: 1395


## 29. Detect duplicate article numbers

Duplicate article numbers inside a supposedly single legal block are treated as a
warning rather than automatically deleted.

Automatic deduplication would be dangerous because two records with the same number
might represent:

- a parser error;
- a repeated/republicated provision;
- a legitimate structural peculiarity;
- two different pieces of text where only one should have been selected.

The correct response is therefore to **flag first, inspect second**.

In [28]:
selected_duplicates = []

grouped_selected = defaultdict(list)

for article in selected_articles:
    grouped_selected[
        (article["document_id"], article["block_id"])
    ].append(article)

for (document_id, block_id), articles in grouped_selected.items():
    counts = Counter(
        article["article"]
        for article in articles
    )

    duplicates = {
        number: count
        for number, count in counts.items()
        if count > 1
    }

    if duplicates:
        selected_duplicates.append({
            "document_id": document_id,
            "block_id": block_id,
            "duplicates": duplicates
        })

        print(
            f"{document_id.upper()} | "
            f"{block_id} | {duplicates}"
        )

print(
    f"\nSelected blocks with duplicate article numbers: "
    f"{len(selected_duplicates)}"
)

CIMI | cimi_block_02 | {'25.º': 2}
CIRC | circ_block_01 | {'73.º': 2, '91.º': 2}
CIRS | cirs_block_02 | {'11.º': 2, '103.º': 2, '128.º': 2}
RGIT | rgit_block_02 | {'30.º': 2, '31.º': 2, '32.º': 2, '40.º': 2, '70.º': 2, '75.º': 2, '79.º': 2, '80.º': 2, '83.º': 2, '84.º': 2, '104.º': 2}
OE2026 | oe2026_block_01 | {'27.º': 2, '52.º': 2, '61.º': 2, '68.º': 2, '70.º': 2, '88.º': 2, '76.º': 2, '94.º': 2, '101.º': 2, '106.º': 2, '108.º': 2, '109.º': 2, '115.º': 2}

Selected blocks with duplicate article numbers: 5


## 30. Create collision-proof source identifiers

Machine-learning datasets require technically unique identifiers even when the source
structure contains anomalies.

We preserve the original logical identifier but add an occurrence suffix when
necessary:

```text
civa_block_03_art_10
civa_block_03_art_10__occ_02
```

This does not claim that the duplicate is legally correct. It merely prevents a
technical ID collision while preserving the anomaly for review.

This distinction between **source truth** and **technical uniqueness** is important in
dataset engineering.

In [29]:
occurrence_counter = Counter()

for article in selected_articles:
    logical_id = article["article_id"]

    occurrence_counter[logical_id] += 1
    occurrence = occurrence_counter[logical_id]

    article["logical_article_id"] = logical_id
    article["article_occurrence"] = occurrence

    if occurrence == 1:
        article["source_article_id"] = logical_id
    else:
        article["source_article_id"] = (
            f"{logical_id}__occ_{occurrence:02d}"
        )


source_ids = [
    article["source_article_id"]
    for article in selected_articles
]

print(f"Source IDs : {len(source_ids)}")
print(f"Unique IDs : {len(set(source_ids))}")

Source IDs : 1395
Unique IDs : 1395


## 31. Detect suspiciously large articles

A very large article may be genuine, but it may also indicate a missed boundary.

For example, if the parser fails to recognise the heading of the next article, all
subsequent text can be appended to the current one.

Articles above the configured character threshold are therefore flagged for manual
inspection rather than automatically removed.

The threshold is a diagnostic heuristic, not a legal rule.

In [30]:
CORPUS_LARGE_ARTICLE_THRESHOLD = 15000

selected_large_articles = []

for article in selected_articles:
    character_count = len(
        article.get("text", "")
    )

    if character_count > CORPUS_LARGE_ARTICLE_THRESHOLD:
        selected_large_articles.append({
            "document_id": article["document_id"],
            "block_id": article["block_id"],
            "article": article["article"],
            "source_article_id": article["source_article_id"],
            "characters": character_count,
            "page_start": article["page_start"],
            "page_end": article["page_end"]
        })

print(
    f"Large selected articles: "
    f"{len(selected_large_articles)}"
)

for item in selected_large_articles:
    print(item)

Large selected articles: 6
{'document_id': 'cirs', 'block_id': 'cirs_block_02', 'article': '10.º', 'source_article_id': 'cirs_block_02_art_10_o', 'characters': 23012, 'page_start': 34, 'page_end': 40}
{'document_id': 'cirs', 'block_id': 'cirs_block_02', 'article': '72.º', 'source_article_id': 'cirs_block_02_art_72_o', 'characters': 19174, 'page_start': 86, 'page_end': 91}
{'document_id': 'civa', 'block_id': 'civa_block_03', 'article': '6.º', 'source_article_id': 'civa_block_03_art_6_o', 'characters': 22955, 'page_start': 81, 'page_end': 87}
{'document_id': 'civa', 'block_id': 'civa_block_03', 'article': '102.º', 'source_article_id': 'civa_block_03_art_102_o', 'characters': 41512, 'page_start': 170, 'page_end': 188}
{'document_id': 'oe2026', 'block_id': 'oe2026_block_01', 'article': '8.º', 'source_article_id': 'oe2026_block_01_art_8_o', 'characters': 18593, 'page_start': 6, 'page_end': 11}
{'document_id': 'oe2026', 'block_id': 'oe2026_block_01', 'article': '262.º', 'source_article_id': 

## 32. Assign a transparent validation status

Every selected article receives an explicit status:

- `ready` — no automatic warning was detected;
- `review_duplicate` — duplicate article number;
- `review_large` — unusually large article;
- `review_duplicate_and_large` — both warnings.

This makes downstream filtering explainable. Instead of silently losing data, we can
always answer:

> Why was this article excluded from the training candidate corpus?

In [31]:
duplicate_keys = set()

for item in selected_duplicates:
    for article_number in item["duplicates"]:
        duplicate_keys.add((
            item["document_id"],
            item["block_id"],
            article_number
        ))

large_ids = {
    item["source_article_id"]
    for item in selected_large_articles
}

for article in selected_articles:
    duplicate = (
        article["document_id"],
        article["block_id"],
        article["article"]
    ) in duplicate_keys

    large = (
        article["source_article_id"]
        in large_ids
    )

    if duplicate and large:
        status = "review_duplicate_and_large"
    elif duplicate:
        status = "review_duplicate"
    elif large:
        status = "review_large"
    else:
        status = "ready"

    article["validation_status"] = status


status_counts = Counter(
    article["validation_status"]
    for article in selected_articles
)

print("Validation status:")
for status, count in status_counts.items():
    print(f" - {status:28}: {count}")

Validation status:
 - ready                       : 1329
 - review_duplicate            : 60
 - review_large                : 6


## 33. Build the clean training-candidate corpus

Only articles with `validation_status == "ready"` enter the clean candidate corpus.

Flagged records remain available in a separate review collection. They can later be
corrected and reintroduced if manual inspection confirms that they are valid.

This creates an important quality gate between document parsing and dataset
construction.

In [32]:
ready_articles = [
    article
    for article in selected_articles
    if article["validation_status"] == "ready"
]

review_articles = [
    article
    for article in selected_articles
    if article["validation_status"] != "ready"
]

print(f"Ready for dataset construction : {len(ready_articles)}")
print(f"Requires manual review         : {len(review_articles)}")

Ready for dataset construction : 1329
Requires manual review         : 66


## 34. Review corpus statistics

Before exporting the final candidate corpus, we summarise the result by legal document.

Useful questions include:

- How many articles were selected?
- How many passed automatically?
- How many require review?
- Is one document producing disproportionately many warnings?
- Does the corpus size broadly match expectations?

Large unexpected differences between documents often reveal upstream parsing or
configuration problems.

In [33]:
corpus_statistics = []

for document_id in CORPUS_CONFIG:
    selected = [
        article
        for article in selected_articles
        if article["document_id"] == document_id
    ]

    ready = [
        article
        for article in selected
        if article["validation_status"] == "ready"
    ]

    review = [
        article
        for article in selected
        if article["validation_status"] != "ready"
    ]

    corpus_statistics.append({
        "document_id": document_id,
        "block_id": CORPUS_CONFIG[document_id]["main_block"],
        "selected_articles": len(selected),
        "ready_articles": len(ready),
        "review_articles": len(review),
        "characters_ready": sum(
            len(article.get("text", ""))
            for article in ready
        )
    })

corpus_statistics

[{'document_id': 'cimi',
  'block_id': 'cimi_block_02',
  'selected_articles': 155,
  'ready_articles': 153,
  'review_articles': 2,
  'characters_ready': 150027},
 {'document_id': 'circ',
  'block_id': 'circ_block_01',
  'selected_articles': 171,
  'ready_articles': 167,
  'review_articles': 4,
  'characters_ready': 394003},
 {'document_id': 'cirs',
  'block_id': 'cirs_block_02',
  'selected_articles': 199,
  'ready_articles': 191,
  'review_articles': 8,
  'characters_ready': 343275},
 {'document_id': 'cis',
  'block_id': 'cis_block_02',
  'selected_articles': 73,
  'ready_articles': 73,
  'review_articles': 0,
  'characters_ready': 108906},
 {'document_id': 'civa',
  'block_id': 'civa_block_03',
  'selected_articles': 102,
  'ready_articles': 100,
  'review_articles': 2,
  'characters_ready': 263688},
 {'document_id': 'ebf',
  'block_id': 'ebf_block_05',
  'selected_articles': 122,
  'ready_articles': 122,
  'review_articles': 0,
  'characters_ready': 258415},
 {'document_id': 'lgt'

## 35. Inspect records requiring manual review

This helper prints the metadata and beginning of every flagged article.

The objective is not merely to inspect the warning label. We need to compare the text
with the source document and determine whether the warning represents a genuine parser
problem or a legitimate legal provision.

If a systematic parser error is found, the preferred solution is to correct the parser
and rerun the pipeline rather than manually editing many downstream records.

In [34]:
def inspect_review_article(
    article,
    max_chars=2500
):
    print("=" * 80)
    print(f"Source ID : {article['source_article_id']}")
    print(f"Document  : {article['document_id']}")
    print(f"Block     : {article['block_id']}")
    print(f"Article   : {article['article']}")
    print(f"Title     : {article.get('article_title')}")
    print(f"Status    : {article['validation_status']}")
    print(
        f"Pages     : "
        f"{article['page_start']}-"
        f"{article['page_end']}"
    )
    print(
        f"Characters: "
        f"{len(article.get('text', ''))}"
    )
    print("\nText:")
    print(article.get("text", "")[:max_chars])


for article in review_articles:
    inspect_review_article(article)

Source ID : cimi_block_02_art_25_o
Document  : cimi
Block     : cimi_block_02
Article   : 25.º
Title     : None
Status    : review_duplicate
Pages     : 34-34
Characters: 267

Text:
2 - O rendimento fundiário de um prédio apura-se a partir da soma dos rendimentos das suas
parcelas com os das árvores dispersas nelas existentes, quando pertencentes ao titular do direito
ao rendimento do prédio e se, no seu conjunto, tiverem interesse económico.
33
Source ID : cimi_block_02_art_25_o__occ_02
Document  : cimi
Block     : cimi_block_02
Article   : 25.º
Title     : Encargos de exploração
Status    : review_duplicate
Pages     : 37-37
Characters: 2422

Text:
A determinação dos encargos prevista no artigo anterior obedece às seguintes regras:
a) O montante dos encargos referidos nas alíneas a) e b) do n.o 1 do artigo 24.o é fixado segundo
os usos locais, tendo em conta os preços de custo dos factores de exploração;
b) As despesas de cultura são as habitualmente correspondentes à cultura propria

## 36. Export the controlled legal corpus

Three JSONL files are created:

```text
legal_corpus_selected.jsonl
    all articles from explicitly selected blocks

legal_corpus_ready.jsonl
    selected articles that passed automatic checks

legal_corpus_review.jsonl
    selected articles requiring manual inspection
```

### Why JSONL?

JSON Lines stores one JSON object per line. It is convenient for machine-learning
pipelines because records can be streamed independently and large files do not need to
be represented as one giant JSON array.

`legal_corpus_ready.jsonl` is the intended input for notebook 03.

In [35]:
SELECTED_PATH = (
    PROCESSED_DIR / "legal_corpus_selected.jsonl"
)

READY_PATH = (
    PROCESSED_DIR / "legal_corpus_ready.jsonl"
)

REVIEW_PATH = (
    PROCESSED_DIR / "legal_corpus_review.jsonl"
)


def write_jsonl(path, records):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


write_jsonl(
    SELECTED_PATH,
    selected_articles
)

write_jsonl(
    READY_PATH,
    ready_articles
)

write_jsonl(
    REVIEW_PATH,
    review_articles
)


CORPUS_REPORT = {
    "parser_version": "5.0",
    "corpus_stage": "6.0",
    "corpus_config": CORPUS_CONFIG,
    "statistics": corpus_statistics,
    "validation_status_counts": dict(status_counts),
    "duplicate_report": selected_duplicates,
    "large_article_report": selected_large_articles
}

REPORT_PATH = (
    PROCESSED_DIR / "_corpus_v6_report.json"
)

with REPORT_PATH.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        CORPUS_REPORT,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Created:")
print(f" - {SELECTED_PATH.resolve()}")
print(f" - {READY_PATH.resolve()}")
print(f" - {REVIEW_PATH.resolve()}")
print(f" - {REPORT_PATH.resolve()}")

Created:
 - C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\legal_corpus_selected.jsonl
 - C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\legal_corpus_ready.jsonl
 - C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\legal_corpus_review.jsonl
 - C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\processed\_corpus_v6_report.json


## 37. Final quality gate before dataset construction

The first two notebooks now have clearly separated responsibilities:

```text
PDF files
   ↓
01 — Extraction
   ↓
page-preserving JSON
   ↓
02 — Parsing
   ↓
legal blocks + hierarchy + articles
   ↓
corpus selection and validation
   ↓
legal_corpus_ready.jsonl
   ↓
03 — Dataset construction
```

Before continuing, inspect `legal_corpus_review.jsonl`.

If a flagged record exposes a genuine parsing problem, correct the parser or corpus
configuration and rerun the pipeline. Avoid deleting problematic records merely to
make the statistics look cleaner.

## Key concepts from this notebook

### Rule-based parsing
Regular expressions and stateful rules can be highly effective when the source domain
has explicit structural conventions.

### Document-level evidence
A line cannot always be classified correctly in isolation. Frequency and page position
help distinguish content from headers and footers.

### Provenance
Every reconstructed article remains connected to its source document, legal block and
page range.

### Diagnostics before training
Dataset errors become model-training errors. Structural validation is therefore part
of the machine-learning pipeline, not an optional cosmetic step.

### Explicit corpus selection
The parser describes what it found; the experiment configuration states what should be
used.

---

## Next step — `03_dataset.ipynb`

The next notebook can focus on converting validated legal articles into examples
suitable for embedding-model training and evaluation.

Conceptually:

```text
validated legal articles
       ↓
queries + relevant passages
       ↓
train / validation / test split
       ↓
qrels and dataset files
```

At that point we move from **document engineering** to **information-retrieval dataset
engineering**.